In [ ]:
import os
import sys
import glob
import cv2
import numpy as np
import pandas as pd
import shutil
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "d:/New folder/Non-Invasive/rPPG"
import csv
import time
import math


## Inlined source

The cells below contain the model / loss source that was previously imported from the `neural_methods/` (and `evaluation/`) packages. They are inlined here so the notebook is self-contained.


In [ ]:
# === inlined from neural_methods/model/RhythmFormer.py ===
""" 
RhythmFormer:Extracting rPPG Signals Based on Hierarchical Temporal Periodic Transformer
"""
from typing import Optional
import torch
from torch import nn, Tensor, LongTensor
from torch.nn import functional as F
import math
from typing import Tuple, Union
from timm.models.layers import trunc_normal_, DropPath



"""
Adapted from here: https://github.com/rayleizhu/BiFormer
"""
import torch
from torch import Tensor, LongTensor , nn
import torch.nn.functional as F
from typing import Optional, Tuple
            
def _grid2seq(x:Tensor, region_size:Tuple[int], num_heads:int):
    """
    Args:
        x: BCTHW tensor
        region size: int
        num_heads: number of attention heads
    Return:
        out: rearranged x, has a shape of (bs, nhead, nregion, reg_size, head_dim)
        region_t, region_h, region_w: number of regions per t/col/row
    """
    B, C, T, H, W = x.size()
    region_t ,region_h, region_w = T//region_size[0],  H//region_size[1],  W//region_size[2]
    x = x.view(B, num_heads, C//num_heads, region_t, region_size[0],region_h, region_size[1], region_w, region_size[2])
    x = torch.einsum('bmdtohpwq->bmthwopqd', x).flatten(2, 4).flatten(-4, -2) # (bs, nhead, nregion, reg_size, head_dim)
    return x, region_t, region_h, region_w


def _seq2grid(x:Tensor, region_t:int, region_h:int, region_w:int, region_size:Tuple[int]):
    """
    Args: 
        x: (bs, nhead, nregion, reg_size^2, head_dim)
    Return:
        x: (bs, C, T, H, W)
    """
    bs, nhead, nregion, reg_size_square, head_dim = x.size()
    x = x.view(bs, nhead, region_t, region_h, region_w, region_size[0], region_size[1], region_size[2], head_dim)
    x = torch.einsum('bmthwopqd->bmdtohpwq', x).reshape(bs, nhead*head_dim,
        region_t*region_size[0],region_h*region_size[1], region_w*region_size[2])
    return x


def video_regional_routing_attention_torch(
    query:Tensor, key:Tensor, value:Tensor, scale:float,
    region_graph:LongTensor, region_size:Tuple[int],
    kv_region_size:Optional[Tuple[int]]=None,
    auto_pad=False)->Tensor:
    """
    Args:
        query, key, value: (B, C, T, H, W) tensor
        scale: the scale/temperature for dot product attention
        region_graph: (B, nhead, t_q*h_q*w_q, topk) tensor, topk <= t_k*h_k*w_k
        region_size: region/window size for queries, (rt, rh, rw)
        key_region_size: optional, if None, key_region_size=region_size
    Return:
        output: (B, C, T, H, W) tensor
        attn: (bs, nhead, q_nregion, reg_size, topk*kv_region_size) attention matrix
    """
    kv_region_size = kv_region_size or region_size
    bs, nhead, q_nregion, topk = region_graph.size()
    
    # # Auto pad to deal with any input size 
    # q_pad_b, q_pad_r, kv_pad_b, kv_pad_r = 0, 0, 0, 0
    # if auto_pad:
    #     _, _, Hq, Wq = query.size()
    #     q_pad_b = (region_size[0] - Hq % region_size[0]) % region_size[0]
    #     q_pad_r = (region_size[1] - Wq % region_size[1]) % region_size[1]
    #     if (q_pad_b > 0 or q_pad_r > 0):
    #         query = F.pad(query, (0, q_pad_r, 0, q_pad_b)) # zero padding

    #     _, _, Hk, Wk = key.size()
    #     kv_pad_b = (kv_region_size[0] - Hk % kv_region_size[0]) % kv_region_size[0]
    #     kv_pad_r = (kv_region_size[1] - Wk % kv_region_size[1]) % kv_region_size[1]
    #     if (kv_pad_r > 0 or kv_pad_b > 0):
    #         key = F.pad(key, (0, kv_pad_r, 0, kv_pad_b)) # zero padding
    #         value = F.pad(value, (0, kv_pad_r, 0, kv_pad_b)) # zero padding
    
    # to sequence format, i.e. (bs, nhead, nregion, reg_size, head_dim)
    query, q_region_t, q_region_h, q_region_w = _grid2seq(query, region_size=region_size, num_heads=nhead)
    key, _, _, _ = _grid2seq(key, region_size=kv_region_size, num_heads=nhead)
    value, _, _, _ = _grid2seq(value, region_size=kv_region_size, num_heads=nhead)

    # gather key and values.
    # torch.gather does not support broadcasting, hence we do it manually
    bs, nhead, kv_nregion, kv_region_size, head_dim = key.size()
    broadcasted_region_graph = region_graph.view(bs, nhead, q_nregion, topk, 1, 1).\
        expand(-1, -1, -1, -1, kv_region_size, head_dim)
    key_g = torch.gather(key.view(bs, nhead, 1, kv_nregion, kv_region_size, head_dim).\
        expand(-1, -1, query.size(2), -1, -1, -1), dim=3,
        index=broadcasted_region_graph) # (bs, nhead, q_nregion, topk, kv_region_size, head_dim)
    value_g = torch.gather(value.view(bs, nhead, 1, kv_nregion, kv_region_size, head_dim).\
        expand(-1, -1, query.size(2), -1, -1, -1), dim=3,
        index=broadcasted_region_graph) # (bs, nhead, q_nregion, topk, kv_region_size, head_dim)
    
    # token-to-token attention
    # (bs, nhead, q_nregion, reg_size, head_dim) @ (bs, nhead, q_nregion, head_dim, topk*kv_region_size)
    # -> (bs, nhead, q_nregion, reg_size, topk*kv_region_size)
    attn = (query * scale) @ key_g.flatten(-3, -2).transpose(-1, -2)
    attn = torch.softmax(attn, dim=-1)
    # (bs, nhead, q_nregion, reg_size, topk*kv_region_size) @ (bs, nhead, q_nregion, topk*kv_region_size, head_dim)
    # -> (bs, nhead, q_nregion, reg_size, head_dim)
    output = attn @ value_g.flatten(-3, -2)

    # to BCTHW format
    output = _seq2grid(output, region_t=q_region_t, region_h=q_region_h, region_w=q_region_w, region_size=region_size)

    # remove paddings if needed
    # if auto_pad and (q_pad_b > 0 or q_pad_r > 0):
    #     output = output[:, :, :Hq, :Wq]

    return output, attn




class CDC_T(nn.Module):
    """
    The CDC_T Module is from here: https://github.com/ZitongYu/PhysFormer/model/transformer_layer.py
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,
                 padding=1, dilation=1, groups=1, bias=False, theta=0.6):

        super(CDC_T, self).__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding,
                              dilation=dilation, groups=groups, bias=bias)
        self.theta = theta

    def forward(self, x):
        out_normal = self.conv(x)

        if math.fabs(self.theta - 0.0) < 1e-8:
            return out_normal
        else:
            # pdb.set_trace()
            [C_out, C_in, t, kernel_size, kernel_size] = self.conv.weight.shape

            # only CD works on temporal kernel size>1
            if self.conv.weight.shape[2] > 1:
                kernel_diff = self.conv.weight[:, :, 0, :, :].sum(2).sum(2) + self.conv.weight[:, :, 2, :, :].sum(
                    2).sum(2)
                kernel_diff = kernel_diff[:, :, None, None, None]
                out_diff = F.conv3d(input=x, weight=kernel_diff, bias=self.conv.bias, stride=self.conv.stride,
                                    padding=0, dilation=self.conv.dilation, groups=self.conv.groups)
                return out_normal - self.theta * out_diff

            else:
                return out_normal
      
class video_BRA(nn.Module):

    def __init__(self, dim, num_heads=8, t_patch=8, qk_scale=None, topk=4,  side_dwconv=3, auto_pad=False, attn_backend='torch'):
        super().__init__()

        self.dim = dim
        self.num_heads = num_heads
        assert self.dim % num_heads == 0, 'dim must be divisible by num_heads!'
        self.head_dim = self.dim // self.num_heads
        self.scale = qk_scale or self.dim ** -0.5 
        self.topk = topk
        self.t_patch = t_patch  # frame of patch
        ################side_dwconv (i.e. LCE in Shunted Transformer)###########
        self.lepe = nn.Conv3d(dim, dim, kernel_size=side_dwconv, stride=1, padding=side_dwconv//2, groups=dim) if side_dwconv > 0 else \
                    lambda x: torch.zeros_like(x)
        ##########################################
        self.qkv_linear = nn.Conv3d(self.dim, 3*self.dim, kernel_size=1)
        self.output_linear = nn.Conv3d(self.dim, self.dim, kernel_size=1)
        self.proj_q = nn.Sequential(
            CDC_T(dim, dim, 3, stride=1, padding=1, groups=1, bias=False, theta=0.2),  
            nn.BatchNorm3d(dim),
        )
        self.proj_k = nn.Sequential(
            CDC_T(dim, dim, 3, stride=1, padding=1, groups=1, bias=False, theta=0.2),  
            nn.BatchNorm3d(dim),
        )
        self.proj_v = nn.Sequential(
            nn.Conv3d(dim, dim, 1, stride=1, padding=0, groups=1, bias=False),
        )
        if attn_backend == 'torch':
            self.attn_fn = video_regional_routing_attention_torch
        else:
            raise ValueError('CUDA implementation is not available yet. Please stay tuned.')

    def forward(self, x:Tensor):

        N, C, T, H, W = x.size()
        t_region = max(4 // self.t_patch , 1)
        region_size = (t_region, H//4 , W//4)

        # STEP 1: linear projection
        q , k , v = self.proj_q(x) , self.proj_k(x) ,self.proj_v(x)

        # STEP 2: pre attention
        q_r = F.avg_pool3d(q.detach(), kernel_size=region_size, ceil_mode=True, count_include_pad=False)
        k_r = F.avg_pool3d(k.detach(), kernel_size=region_size, ceil_mode=True, count_include_pad=False) # ncthw
        q_r:Tensor = q_r.permute(0, 2, 3, 4, 1).flatten(1, 3) # n(thw)c
        k_r:Tensor = k_r.flatten(2, 4) # nc(thw)
        a_r = q_r @ k_r # n(thw)(thw)
        _, idx_r = torch.topk(a_r, k=self.topk, dim=-1) # n(thw)k
        idx_r:LongTensor = idx_r.unsqueeze_(1).expand(-1, self.num_heads, -1, -1) 

        # STEP 3: refined attention
        output, attn_mat = self.attn_fn(query=q, key=k, value=v, scale=self.scale,
                                        region_graph=idx_r, region_size=region_size)
        
        output = output + self.lepe(v) # nctHW
        output = self.output_linear(output) # nctHW

        return output

class video_BiFormerBlock(nn.Module):
    def __init__(self, dim, drop_path=0., num_heads=4, t_patch=1,qk_scale=None, topk=4, mlp_ratio=2, side_dwconv=5):
        super().__init__()
        self.t_patch = t_patch
        self.norm1 = nn.BatchNorm3d(dim)
        self.attn = video_BRA(dim=dim, num_heads=num_heads, t_patch=t_patch,qk_scale=qk_scale, topk=topk, side_dwconv=side_dwconv)
        self.norm2 = nn.BatchNorm3d(dim)
        self.mlp = nn.Sequential(nn.Conv3d(dim, int(mlp_ratio*dim), kernel_size=1),
                                 nn.BatchNorm3d(int(mlp_ratio*dim)),
                                 nn.GELU(),
                                 nn.Conv3d(int(mlp_ratio*dim),  int(mlp_ratio*dim), 3, stride=1, padding=1),  
                                 nn.BatchNorm3d(int(mlp_ratio*dim)),
                                 nn.GELU(),
                                 nn.Conv3d(int(mlp_ratio*dim), dim, kernel_size=1),
                                 nn.BatchNorm3d(dim),
                                )
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

class Fusion_Stem(nn.Module):
    def __init__(self,apha=0.5,belta=0.5):
        super(Fusion_Stem, self).__init__()

        self.stem11 = nn.Sequential(nn.Conv2d(3, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
            )
        
        self.stem12 = nn.Sequential(nn.Conv2d(12, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
            )

        self.stem21 =nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.stem22 =nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.apha = apha
        self.belta = belta

    def forward(self, x):
        """Definition of Fusion_Stem.
        Args:
          x [N,D,C,H,W]
        Returns:
          fusion_x [N*D,C,H/4,W/4]
        """
        N, D, C, H, W = x.shape
        x1 = torch.cat([x[:,:1,:,:,:],x[:,:1,:,:,:],x[:,:D-2,:,:,:]],1)
        x2 = torch.cat([x[:,:1,:,:,:],x[:,:D-1,:,:,:]],1)
        x3 = x
        x4 = torch.cat([x[:,1:,:,:,:],x[:,D-1:,:,:,:]],1)
        x5 = torch.cat([x[:,2:,:,:,:],x[:,D-1:,:,:,:],x[:,D-1:,:,:,:]],1)
        x_diff = self.stem12(torch.cat([x2-x1,x3-x2,x4-x3,x5-x4],2).view(N * D, 12, H, W))
        x3 = x3.contiguous().view(N * D, C, H, W)
        x = self.stem11(x3)

        #fusion layer1
        x_path1 = self.apha*x + self.belta*x_diff
        x_path1 = self.stem21(x_path1)
        #fusion layer2
        x_path2 = self.stem22(x_diff)
        x = self.apha*x_path1 + self.belta*x_path2
        
        return x
    
class TPT_Block(nn.Module):
    def __init__(self, dim, depth, num_heads, t_patch, topk,
                 mlp_ratio=4., drop_path=0., side_dwconv=5):
        super().__init__()
        self.dim = dim
        self.depth = depth
        ############ downsample layers & upsample layers #####################
        self.downsample_layers = nn.ModuleList()
        self.upsample_layers = nn.ModuleList()
        self.layer_n = int(math.log(t_patch,2))
        for i in range(self.layer_n):
            downsample_layer = nn.Sequential(
                nn.BatchNorm3d(dim), 
                nn.Conv3d(dim , dim , kernel_size=(2, 1, 1), stride=(2, 1, 1)),
                )
            self.downsample_layers.append(downsample_layer)
            upsample_layer = nn.Sequential(
                nn.Upsample(scale_factor=(2, 1, 1)),
                nn.Conv3d(dim , dim , [3, 1, 1], stride=1, padding=(1, 0, 0)),   
                nn.BatchNorm3d(dim),
                nn.ELU(),
                )
            self.upsample_layers.append(upsample_layer)
        ######################################################################
        self.blocks = nn.ModuleList([
            video_BiFormerBlock(
                    dim=dim,
                    drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                    num_heads=num_heads,
                    t_patch=t_patch,
                    topk=topk,
                    mlp_ratio=mlp_ratio,
                    side_dwconv=side_dwconv,
                )
            for i in range(depth)
        ])
    def forward(self, x:torch.Tensor):
        """Definition of TPT_Block.
        Args:
          x [N,C,D,H,W]
        Returns:
          x [N,C,D,H,W]
        """
        for i in range(self.layer_n) :
            x = self.downsample_layers[i](x)
        for blk in self.blocks:
            x = blk(x)
        for i in range(self.layer_n) :
            x = self.upsample_layers[i](x)

        return x
    
class RhythmFormer(nn.Module):

    def __init__(
        self, 
        name: Optional[str] = None, 
        pretrained: bool = False, 
        dim: int = 64, frame: int = 160,
        image_size: Optional[int] = (160,128,128),
        in_chans=64, head_dim=16,
        stage_n = 3,
        embed_dim=[64, 64, 64], mlp_ratios=[1.5, 1.5, 1.5],
        depth=[2, 2, 2], 
        t_patchs:Union[int, Tuple[int]]=(2, 4, 8),
        topks:Union[int, Tuple[int]]=(40, 40, 40),
        side_dwconv:int=3,
        drop_path_rate=0.,
        use_checkpoint_stages=[],
    ):
        super().__init__()

        self.image_size = image_size  
        self.frame = frame  
        self.dim = dim              
        self.stage_n = stage_n

        self.Fusion_Stem = Fusion_Stem()
        self.patch_embedding = nn.Conv3d(in_chans,embed_dim[0], kernel_size=(1, 4, 4), stride=(1, 4, 4))
        self.ConvBlockLast = nn.Conv1d(embed_dim[-1], 1, kernel_size=1,stride=1, padding=0)

        ##########################################################################
        self.stages = nn.ModuleList()
        nheads= [dim // head_dim for dim in embed_dim]
        dp_rates=[x.item() for x in torch.linspace(0, drop_path_rate, sum(depth))]
        for i in range(stage_n):
            stage = TPT_Block(dim=embed_dim[i],
                               depth=depth[i],
                               num_heads=nheads[i], 
                               mlp_ratio=mlp_ratios[i],
                               drop_path=dp_rates[sum(depth[:i]):sum(depth[:i+1])],
                               t_patch=t_patchs[i], topk=topks[i], side_dwconv=side_dwconv
                               )
            self.stages.append(stage)
        ##########################################################################

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        N, D, C, H, W = x.shape
        x = self.Fusion_Stem(x)    #[N*D 64 H/4 W/4]
        x = x.view(N,D,64,H//4,W//4).permute(0,2,1,3,4)
        x = self.patch_embedding(x)    #[N 64 D 8 8]
        for i in range(3):
            x = self.stages[i](x)    #[N 64 D 8 8]
        features_last = torch.mean(x,3)    #[N, 64, D, 8]  
        features_last = torch.mean(features_last,3)    #[N, 64, D]  
        rPPG = self.ConvBlockLast(features_last)    #[N, 1, D]
        rPPG = rPPG.squeeze(1)
        return rPPG 


In [ ]:
# === inlined from evaluation/post_process.py ===
"""The post processing files for caluclating heart rate using FFT or peak detection.
The file also  includes helper funcs such as detrend, power2db etc.
"""

import numpy as np
import scipy
import scipy.io
from scipy.signal import butter
from scipy.sparse import spdiags
from copy import deepcopy

def _next_power_of_2(x):
    """Calculate the nearest power of 2."""
    return 1 if x == 0 else 2 ** (x - 1).bit_length()

def _detrend(input_signal, lambda_value):
    """Detrend PPG signal."""
    signal_length = input_signal.shape[0]
    # observation matrix
    H = np.identity(signal_length)
    ones = np.ones(signal_length)
    minus_twos = -2 * np.ones(signal_length)
    diags_data = np.array([ones, minus_twos, ones])
    diags_index = np.array([0, 1, 2])
    D = spdiags(diags_data, diags_index,
                (signal_length - 2), signal_length).toarray()
    detrended_signal = np.dot(
        (H - np.linalg.inv(H + (lambda_value ** 2) * np.dot(D.T, D))), input_signal)
    return detrended_signal

def power2db(mag):
    """Convert power to db."""
    return 10 * np.log10(mag)

def _calculate_fft_hr(ppg_signal, fs=60, low_pass=0.6, high_pass=3.3):
    # Note: to more closely match results in the NeurIPS 2023 toolbox paper,
    # we recommend low_pass=0.75 and high_pass=2.5 instead of the defaults above.
    """Calculate heart rate based on PPG using Fast Fourier transform (FFT)."""
    ppg_signal = np.expand_dims(ppg_signal, 0)
    N = _next_power_of_2(ppg_signal.shape[1])
    f_ppg, pxx_ppg = scipy.signal.periodogram(ppg_signal, fs=fs, nfft=N, detrend=False)
    fmask_ppg = np.argwhere((f_ppg >= low_pass) & (f_ppg <= high_pass))
    mask_ppg = np.take(f_ppg, fmask_ppg)
    mask_pxx = np.take(pxx_ppg, fmask_ppg)
    fft_hr = np.take(mask_ppg, np.argmax(mask_pxx, 0))[0] * 60
    return fft_hr

def _calculate_peak_hr(ppg_signal, fs):
    """Calculate heart rate based on PPG using peak detection."""
    ppg_peaks, _ = scipy.signal.find_peaks(ppg_signal)
    hr_peak = 60 / (np.mean(np.diff(ppg_peaks)) / fs)
    return hr_peak

def _compute_macc(pred_signal, gt_signal):
    """Calculate maximum amplitude of cross correlation (MACC) by computing correlation at all time lags.
        Args:
            pred_ppg_signal(np.array): predicted PPG signal 
            label_ppg_signal(np.array): ground truth, label PPG signal
        Returns:
            MACC(float): Maximum Amplitude of Cross-Correlation
    """
    pred = deepcopy(pred_signal)
    gt = deepcopy(gt_signal)
    pred = np.squeeze(pred)
    gt = np.squeeze(gt)
    min_len = np.min((len(pred), len(gt)))
    pred = pred[:min_len]
    gt = gt[:min_len]
    lags = np.arange(0, len(pred)-1, 1)
    tlcc_list = []
    for lag in lags:
        cross_corr = np.abs(np.corrcoef(
            pred, np.roll(gt, lag))[0][1])
        tlcc_list.append(cross_corr)
    macc = max(tlcc_list)
    return macc

def _calculate_SNR(pred_ppg_signal, hr_label, fs=30, low_pass=0.6, high_pass=3.3):
    """Calculate SNR as the ratio of the area under the curve of the frequency spectrum around the first and second harmonics 
        of the ground truth HR frequency to the area under the curve of the remainder of the frequency spectrum, from 0.6 Hz
        to 3.3 Hz. 

        Ref for low_pass and high_pass filters:
        R. Cassani, A. Tiwari and T. H. Falk, "Optimal filter characterization for photoplethysmography-based pulse rate and 
        pulse power spectrum estimation," 2020 IEEE Engineering in Medicine & Biology Society (EMBC), Montreal, QC, Canada,
        doi: 10.1109/EMBC44109.2020.9175396.

        Note: to more closely match results in the NeurIPS 2023 toolbox paper, we recommend low_pass=0.75 and high_pass=2.5 
        instead of the defaults above.

        Args:
            pred_ppg_signal(np.array): predicted PPG signal 
            label_ppg_signal(np.array): ground truth, label PPG signal
            fs(int or float): sampling rate of the video
        Returns:
            SNR(float): Signal-to-Noise Ratio
    """
    # Get the first and second harmonics of the ground truth HR in Hz
    first_harmonic_freq = hr_label / 60
    second_harmonic_freq = 2 * first_harmonic_freq
    deviation = 6 / 60  # 6 beats/min converted to Hz (1 Hz = 60 beats/min)

    # Calculate FFT
    pred_ppg_signal = np.expand_dims(pred_ppg_signal, 0)
    N = _next_power_of_2(pred_ppg_signal.shape[1])
    f_ppg, pxx_ppg = scipy.signal.periodogram(pred_ppg_signal, fs=fs, nfft=N, detrend=False)

    # Calculate the indices corresponding to the frequency ranges
    idx_harmonic1 = np.argwhere((f_ppg >= (first_harmonic_freq - deviation)) & (f_ppg <= (first_harmonic_freq + deviation)))
    idx_harmonic2 = np.argwhere((f_ppg >= (second_harmonic_freq - deviation)) & (f_ppg <= (second_harmonic_freq + deviation)))
    idx_remainder = np.argwhere((f_ppg >= low_pass) & (f_ppg <= high_pass) \
     & ~((f_ppg >= (first_harmonic_freq - deviation)) & (f_ppg <= (first_harmonic_freq + deviation))) \
     & ~((f_ppg >= (second_harmonic_freq - deviation)) & (f_ppg <= (second_harmonic_freq + deviation))))

    # Select the corresponding values from the periodogram
    pxx_ppg = np.squeeze(pxx_ppg)
    pxx_harmonic1 = pxx_ppg[idx_harmonic1]
    pxx_harmonic2 = pxx_ppg[idx_harmonic2]
    pxx_remainder = pxx_ppg[idx_remainder]

    # Calculate the signal power
    signal_power_hm1 = np.sum(pxx_harmonic1)
    signal_power_hm2 = np.sum(pxx_harmonic2)
    signal_power_rem = np.sum(pxx_remainder)

    # Calculate the SNR as the ratio of the areas
    if not signal_power_rem == 0: # catches divide by 0 runtime warning 
        SNR = power2db((signal_power_hm1 + signal_power_hm2) / signal_power_rem)
    else:
        SNR = 0
    return SNR

def calculate_metric_per_video(predictions, labels, fs=30, diff_flag=True, use_bandpass=True, hr_method='FFT'):
    """Calculate video-level HR and SNR"""
    if diff_flag:  # if the predictions and labels are 1st derivative of PPG signal.
        predictions = _detrend(np.cumsum(predictions), 100)
        labels = _detrend(np.cumsum(labels), 100)
    else:
        predictions = _detrend(predictions, 100)
        labels = _detrend(labels, 100)
    if use_bandpass:
        # bandpass filter between [0.75, 2.5] Hz, equals [45, 150] beats per min
        # bandpass filter between [0.6, 3.3] Hz, equals [36, 198] beats per min
        #
        # Note: to more closely match results in the NeurIPS 2023 toolbox paper,
        # we recommend using 0.75 in place of 0.6 and 2.5 in place of 3.3 in the 
        # below line.
        [b, a] = butter(1, [0.6 / fs * 2, 3.3 / fs * 2], btype='bandpass')
        predictions = scipy.signal.filtfilt(b, a, np.double(predictions))
        labels = scipy.signal.filtfilt(b, a, np.double(labels))
    
    macc = _compute_macc(predictions, labels)

    if hr_method == 'FFT':
        hr_pred = _calculate_fft_hr(predictions, fs=fs)
        hr_label = _calculate_fft_hr(labels, fs=fs)
    elif hr_method == 'Peak':
        hr_pred = _calculate_peak_hr(predictions, fs=fs)
        hr_label = _calculate_peak_hr(labels, fs=fs)
    else:
        raise ValueError('Please use FFT or Peak to calculate your HR.')
    SNR = _calculate_SNR(predictions, hr_label, fs=fs)
    return hr_label, hr_pred, SNR, macc

In [ ]:
# === inlined from neural_methods/loss/RythmFormerLossComputer.py ===
'''
  Adapted from here: https://github.com/ZitongYu/PhysFormer/TorchLossComputer.py
  Modifed based on the HR-CNN here: https://github.com/radimspetlik/hr-cnn
'''
import math
import torch
from torch.autograd import Variable
import numpy as np
import torch.nn.functional as F
import torch.nn as nn


def normal_sampling(mean, label_k, std):
    return math.exp(-(label_k-mean)**2/(2*std**2))/(math.sqrt(2*math.pi)*std)

def kl_loss(inputs, labels):
    criterion = nn.KLDivLoss(reduce=False)
    outputs = torch.log(inputs)
    loss = criterion(outputs, labels)
    #loss = loss.sum()/loss.shape[0]
    loss = loss.sum()
    return loss

class Neg_Pearson(nn.Module):    # Pearson range [-1, 1] so if < 0, abs|loss| ; if >0, 1- loss
    def __init__(self):
        super(Neg_Pearson,self).__init__()

    def forward(self, preds, labels):       # all variable operation
        loss = 0
        for i in range(preds.shape[0]):
            sum_x = torch.sum(preds[i])                # x
            sum_y = torch.sum(labels[i])               # y
            sum_xy = torch.sum(preds[i]*labels[i])        # xy
            sum_x2 = torch.sum(torch.pow(preds[i],2))  # x^2
            sum_y2 = torch.sum(torch.pow(labels[i],2)) # y^2
            N = preds.shape[1]
            pearson = (N*sum_xy - sum_x*sum_y)/(torch.sqrt((N*sum_x2 - torch.pow(sum_x,2))*(N*sum_y2 - torch.pow(sum_y,2))))
            loss += 1 - pearson
            
        loss = loss/preds.shape[0]
        return loss

class RhythmFormer_Loss(nn.Module): 
    def __init__(self):
        super(RhythmFormer_Loss,self).__init__()
        self.criterion_Pearson = Neg_Pearson()
    def forward(self, pred_ppg, labels , epoch , FS , diff_flag):    
        loss_time = self.criterion_Pearson(pred_ppg.view(1,-1) , labels.view(1,-1))    
        loss_CE , loss_distribution_kl = TorchLossComputer.Frequency_loss(pred_ppg.squeeze(-1),  labels.squeeze(-1), diff_flag=diff_flag, Fs=FS, std=3.0)
        loss_hr = TorchLossComputer.HR_loss(pred_ppg.squeeze(-1),  labels.squeeze(-1), diff_flag=diff_flag, Fs=FS, std=3.0)
        if torch.isnan(loss_time) : 
           loss_time = 0

        loss = 0.2 * loss_time + 1.0 * loss_CE + 1.0 * loss_hr
        return loss

class TorchLossComputer(object):
    @staticmethod
    def compute_complex_absolute_given_k(output, k, N):
        two_pi_n_over_N = Variable(2 * math.pi * torch.arange(0, N, dtype=torch.float), requires_grad=True) / N
        hanning = Variable(torch.from_numpy(np.hanning(N)).type(torch.FloatTensor), requires_grad=True).view(1, -1)

        k = k.type(torch.FloatTensor).cuda()
        two_pi_n_over_N = two_pi_n_over_N.cuda()
        hanning = hanning.cuda()
            
        output = output.view(1, -1) * hanning
        output = output.view(1, 1, -1).type(torch.cuda.FloatTensor)
        k = k.view(1, -1, 1)
        two_pi_n_over_N = two_pi_n_over_N.view(1, 1, -1)
        complex_absolute = torch.sum(output * torch.sin(k * two_pi_n_over_N), dim=-1) ** 2 \
                           + torch.sum(output * torch.cos(k * two_pi_n_over_N), dim=-1) ** 2

        return complex_absolute

    @staticmethod
    def complex_absolute(output, Fs, bpm_range=None):
        output = output.view(1, -1)

        N = output.size()[1]

        unit_per_hz = Fs / N
        feasible_bpm = bpm_range / 60.0
        k = feasible_bpm / unit_per_hz

        # only calculate feasible PSD range [0.7,4]Hz
        complex_absolute = TorchLossComputer.compute_complex_absolute_given_k(output, k, N)

        return (1.0 / complex_absolute.sum()) * complex_absolute	# Analogous Softmax operator


    @staticmethod
    def cross_entropy_power_spectrum_loss(inputs, target, Fs):
        inputs = inputs.view(1, -1)
        target = target.view(1, -1)
        bpm_range = torch.arange(40, 180, dtype=torch.float).cuda()
        #bpm_range = torch.arange(40, 260, dtype=torch.float).cuda()

        complex_absolute = TorchLossComputer.complex_absolute(inputs, Fs, bpm_range)

        whole_max_val, whole_max_idx = complex_absolute.view(-1).max(0)
        whole_max_idx = whole_max_idx.type(torch.float)
        
        #pdb.set_trace()

        #return F.cross_entropy(complex_absolute, target.view((1)).type(torch.long)).view(1),  (target.item() - whole_max_idx.item()) ** 2
        return F.cross_entropy(complex_absolute, target.view((1)).type(torch.long)),  torch.abs(target[0] - whole_max_idx)

    @staticmethod
    def cross_entropy_power_spectrum_focal_loss(inputs, target, Fs, gamma):
        inputs = inputs.view(1, -1)
        target = target.view(1, -1)
        bpm_range = torch.arange(40, 180, dtype=torch.float).cuda()
        #bpm_range = torch.arange(40, 260, dtype=torch.float).cuda()

        complex_absolute = TorchLossComputer.complex_absolute(inputs, Fs, bpm_range)

        whole_max_val, whole_max_idx = complex_absolute.view(-1).max(0)
        whole_max_idx = whole_max_idx.type(torch.float)
        
        #pdb.set_trace()
        criterion = FocalLoss(gamma=gamma)

        #return F.cross_entropy(complex_absolute, target.view((1)).type(torch.long)).view(1),  (target.item() - whole_max_idx.item()) ** 2
        return criterion(complex_absolute, target.view((1)).type(torch.long)),  torch.abs(target[0] - whole_max_idx)


    @staticmethod
    def cross_entropy_power_spectrum_forward_pred(inputs, Fs):
        inputs = inputs.view(1, -1)
        bpm_range = torch.arange(40, 190, dtype=torch.float).cuda()
        #bpm_range = torch.arange(40, 180, dtype=torch.float).cuda()
        #bpm_range = torch.arange(40, 260, dtype=torch.float).cuda()

        complex_absolute = TorchLossComputer.complex_absolute(inputs, Fs, bpm_range)

        whole_max_val, whole_max_idx = complex_absolute.view(-1).max(0)
        whole_max_idx = whole_max_idx.type(torch.float)

        return whole_max_idx

    @staticmethod
    def Frequency_loss(inputs, target, diff_flag , Fs, std):
        hr_gt, pred_hr_peak, SNR, macc = calculate_metric_per_video(inputs.detach().cpu(), target.detach().cpu(), diff_flag = diff_flag, fs=Fs, hr_method='FFT')
        inputs = inputs.view(1, -1)
        target = target.view(1, -1)
        bpm_range = torch.arange(45, 150, dtype=torch.float).to(torch.device('cuda'))
        ca = TorchLossComputer.complex_absolute(inputs, Fs, bpm_range)
        sa = ca/torch.sum(ca)

        target_distribution = [normal_sampling(int(hr_gt), i, std) for i in range(45, 150)]
        target_distribution = [i if i > 1e-15 else 1e-15 for i in target_distribution]
        target_distribution = torch.Tensor(target_distribution).to(torch.device('cuda'))

        hr_gt = torch.tensor(hr_gt-45).view(1).type(torch.long).to(torch.device('cuda'))
        return F.cross_entropy(ca, hr_gt) , kl_loss(sa , target_distribution)

    @staticmethod
    def HR_loss(inputs, target,  diff_flag , Fs, std):
        psd_gt, psd_pred, SNR, macc = calculate_metric_per_video(inputs.detach().cpu(), target.detach().cpu(), diff_flag = diff_flag, fs=Fs, hr_method='Peak')
        pred_distribution = [normal_sampling(np.argmax(psd_pred), i, std) for i in range(psd_pred.size)]
        pred_distribution = [i if i > 1e-15 else 1e-15 for i in pred_distribution]
        pred_distribution = torch.Tensor(pred_distribution).to(torch.device('cuda'))
        target_distribution = [normal_sampling(np.argmax(psd_gt), i, std) for i in range(psd_gt.size)]
        target_distribution = [i if i > 1e-15 else 1e-15 for i in target_distribution]
        target_distribution = torch.Tensor(target_distribution).to(torch.device('cuda'))
        return kl_loss(pred_distribution , target_distribution)


## Training utilities

Seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.


In [ ]:
# === Training utilities (shared across notebooks) ===
# seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.
import os
import random as _random
import csv as _csv
import time as _time
import numpy as _np
import torch as _torch
from scipy.signal import periodogram as _periodogram


def set_seed(seed: int = 42):
    """Set seeds for reproducibility across random, numpy, torch (CPU + CUDA)."""
    _random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)
    _torch.cuda.manual_seed_all(seed)
    _torch.backends.cudnn.deterministic = True
    _torch.backends.cudnn.benchmark = False


def train_val_split(dataset, val_ratio: float = 0.2, seed: int = 42):
    """Random split returning (train_subset, val_subset) with a seeded generator."""
    n_total = len(dataset)
    n_val = max(1, int(n_total * val_ratio))
    n_train = n_total - n_val
    g = _torch.Generator().manual_seed(seed)
    return _torch.utils.data.random_split(dataset, [n_train, n_val], generator=g)


def compute_hr_fft(signal_1d, fps: int = 30, lo_hz: float = 0.6, hi_hz: float = 3.3) -> float:
    """Peak-frequency HR (bpm) of a 1-D signal via periodogram, restricted to a band."""
    sig = signal_1d.detach().cpu().numpy() if _torch.is_tensor(signal_1d) else _np.asarray(signal_1d)
    sig = sig.astype(_np.float64).ravel()
    if sig.size < 8 or sig.std() < 1e-8:
        return 0.0
    sig = sig - sig.mean()
    freqs, psd = _periodogram(sig, fs=fps)
    band = (freqs >= lo_hz) & (freqs <= hi_hz)
    if not band.any():
        return 0.0
    return float(freqs[band][psd[band].argmax()] * 60.0)


def compute_hr_mae_batch(preds, labels, fps: int = 30) -> float:
    """Mean absolute HR error (bpm) over a batch.  preds/labels: (N, T) or (T,)."""
    if preds.dim() == 1:
        preds = preds.unsqueeze(0)
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    errs = []
    for i in range(preds.shape[0]):
        hr_p = compute_hr_fft(preds[i], fps)
        hr_l = compute_hr_fft(labels[i], fps)
        errs.append(abs(hr_p - hr_l))
    return float(_np.mean(errs)) if errs else 0.0


class BestCheckpointSaver:
    """Save the model's state_dict whenever a tracked metric improves."""
    def __init__(self, path: str, mode: str = "min"):
        assert mode in ("min", "max")
        self.path = path
        self.mode = mode
        self.best = float("inf") if mode == "min" else -float("inf")
        os.makedirs(os.path.dirname(os.path.abspath(path)) or ".", exist_ok=True)

    def step(self, model, metric: float) -> bool:
        improved = (metric < self.best) if self.mode == "min" else (metric > self.best)
        if improved and not (metric != metric):  # reject NaN
            self.best = metric
            _torch.save(model.state_dict(), self.path)
            return True
        return False


class EarlyStopping:
    """Stop training when the tracked metric stops improving for `patience` epochs."""
    def __init__(self, patience: int = 5, mode: str = "min", min_delta: float = 0.0):
        assert mode in ("min", "max")
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = float("inf") if mode == "min" else -float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metric: float) -> bool:
        improved = (
            (self.mode == "min" and metric < self.best - self.min_delta)
            or (self.mode == "max" and metric > self.best + self.min_delta)
        )
        if improved:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


class MetricLogger:
    """Append per-epoch metrics to a CSV.  Creates the file with headers on init."""
    def __init__(self, csv_path: str, fieldnames=None):
        self.csv_path = csv_path
        self.fieldnames = list(fieldnames) if fieldnames else [
            "epoch", "train_loss", "val_loss", "val_hr_mae", "lr", "time_sec"
        ]
        os.makedirs(os.path.dirname(os.path.abspath(csv_path)) or ".", exist_ok=True)
        with open(csv_path, "w", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writeheader()

    def log(self, **row):
        with open(self.csv_path, "a", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writerow(
                {k: row.get(k, "") for k in self.fieldnames}
            )


# Seed the run for reproducibility
set_seed(42)
print("Training utils ready  |  seed=42  |  HR-MAE band: 0.6-3.3 Hz (36-198 bpm)")


In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupG")
OUTPUT_MODEL_PATH   = os.path.join(REPO_ROOT, "final_model_release/GroupG_RhythmFormer.pth")

# ----- params -----
VIDEO_FPS    = 30
CHUNK_LENGTH = 160
IMG_H, IMG_W = 128, 128
BATCH_SIZE   = 4
EPOCHS       = 30
LR           = 3e-4

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_MODEL_PATH), exist_ok=True)

In [ ]:
def read_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.stack(frames, axis=0)

def read_ppg_synced(session_path, num_frames):
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    min_len = min(len(frame_df), num_frames)
    frame_t = frame_df["timestamp"].values[:min_len]
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    return ppg_resampled.astype(np.float32)

def standardized_data(data):
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0: data = (data - m) / s
    else: data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data

def standardized_label(label):
    label = label.astype(np.float64)
    m = np.mean(label)
    s = np.std(label)
    if s > 0: label = (label - m) / s
    else: label = np.zeros_like(label)
    return label.astype(np.float32)

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)
    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]
    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H
    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h), interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
all_dirs = sorted([d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*")) if os.path.isdir(d) and os.path.basename(d) != "videos"])
subjects = []
for subj_dir in all_dirs:
    subj_id = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")
    video_files = glob.glob(os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv"))
    if not video_files: continue
    subjects.append({"subj_id": subj_id, "subj_key": subj_key, "video_path": video_files[0], "session_path": subj_dir})

print(f"Total subjects: {len(subjects)}")

# Preprocessing
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)

all_input_files = []
for subj in subjects:
    subj_key = subj["subj_key"]
    frames = read_video_frames(subj["video_path"])
    T = frames.shape[0]
    ppg_signal = read_ppg_synced(subj["session_path"], T)
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)
    data_normalized = standardized_data(frames_cropped)
    label_normalized = standardized_label(ppg_signal)
    
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([data_normalized[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]  for i in range(clip_num)])
    label_clips = np.array([label_normalized[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir, exist_ok=True)
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")
        np.save(input_path, data_clips[chunk_idx])
        np.save(label_path, label_clips[chunk_idx])
        all_input_files.append(input_path)

print(f"Total clips saved: {len(all_input_files)}")

In [ ]:
class RhythmFormerDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [f.replace("input", "label") for f in self.inputs]
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))
        label = np.float32(np.load(self.labels[index]))
        data = np.transpose(data, (0, 3, 1, 2))  # NDHWC -> NDCHW
        return data, label

# Train/val split (seeded) + DataLoaders
SEED = 42
VAL_RATIO = 0.2
PATIENCE = 5

train_ds, val_ds = train_val_split(dataset, val_ratio=VAL_RATIO, seed=SEED)
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)} ({len(train_loader)} batches)  |  Val: {len(val_ds)} ({len(val_loader)} batches)")


In [ ]:
# Optimized RhythmFormer training (val/best/early-stop/grad-clip/CSV log)
LOG_PATH = os.path.join(REPO_ROOT, "results/Headmotion/groupG/train_logs/RhythmFormer.csv")

model = RhythmFormer().to(DEVICE)
criterion = RhythmFormer_Loss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=max(1, len(train_loader))
)
saver   = BestCheckpointSaver(OUTPUT_MODEL_PATH, mode="min")
stopper = EarlyStopping(patience=PATIENCE, mode="min")
logger  = MetricLogger(LOG_PATH)
diff_flag = 0  # LABEL_TYPE = "Standardized"


def _norm_per_clip(x):
    return (x - x.mean(dim=-1, keepdim=True)) / (x.std(dim=-1, keepdim=True) + 1e-7)


for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    train_loss_sum, n_train = 0.0, 0
    tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", ncols=90)
    for batch in tbar:
        data, labels = batch[0].float().to(DEVICE, non_blocking=True), batch[1].float().to(DEVICE, non_blocking=True)
        N = data.shape[0]
        optimizer.zero_grad()
        pred = model(data)
        pred = _norm_per_clip(pred)
        # RhythmFormer_Loss expects 1-D signals -> loop unavoidable here
        loss = sum(criterion(pred[ib], labels[ib], epoch, VIDEO_FPS, diff_flag) for ib in range(N)) / N
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss_sum += loss.item()
        n_train += 1
        tbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss = train_loss_sum / max(1, n_train)

    model.eval()
    val_loss_sum, val_hr_mae_sum, n_val = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  ", ncols=90):
            data, labels = batch[0].float().to(DEVICE, non_blocking=True), batch[1].float().to(DEVICE, non_blocking=True)
            N = data.shape[0]
            pred = model(data)
            pred = _norm_per_clip(pred)
            loss = sum(criterion(pred[ib], labels[ib], epoch, VIDEO_FPS, diff_flag) for ib in range(N)) / N
            val_loss_sum += loss.item()
            val_hr_mae_sum += compute_hr_mae_batch(pred, labels, fps=VIDEO_FPS) * N
            n_val += N
    val_loss   = val_loss_sum / max(1, len(val_loader))
    val_hr_mae = val_hr_mae_sum / max(1, n_val)
    elapsed = time.time() - t0
    cur_lr = optimizer.param_groups[0]["lr"]

    improved = saver.step(model, val_hr_mae)
    marker = " <- best" if improved else ""
    print(f"Epoch {epoch+1:2d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_HR_MAE={val_hr_mae:.2f} bpm  lr={cur_lr:.2e}  ({elapsed:.1f}s){marker}")
    logger.log(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
               val_hr_mae=val_hr_mae, lr=cur_lr, time_sec=elapsed)

    if stopper.step(val_hr_mae):
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest val HR-MAE: {saver.best:.2f} bpm  ->  {OUTPUT_MODEL_PATH}")
